# Marketing Agent

WHAT YOU'RE BUILDING: a multi-step marketing agent that turns a product +
audience segment into a structured campaign PLAN and a presentation DECK.

SUCCESS CRITERIA (ANWB): a structured marketing plan; a presentation output;
a demonstrable agent workflow; visible multi-step reasoning and execution.

HOW TO RUN - two ways, same result:
  SQL      : paste this file into a Snowsight worksheet and click Run All
             (or run: snow sql -c <connection> -f marketing_agent.sql).
  Notebook : import marketing_agent.ipynb into Snowsight and Run All.
After Run All, follow the SNOWSIGHT STEPS at the bottom to build the agent
(and, optionally, the app).

Self-contained and idempotent; independent of Challenge 1. The deck defaults
to HTML (no extra packages); a real .pptx is an optional stretch that needs
python-pptx (accept Anaconda terms in Snowsight > Admin > Billing & Terms).
Web Search for live market context is a built-in Cortex Agent tool (not a
custom integration): an admin enables it once for the account
(ALTER ACCOUNT SET ENABLE_CORTEX_WEBSEARCH = true, or AI & ML > Agents >
Settings > Web search), then you add the Web Search tool to the agent.

The agent gets three kinds of grounding: (a) RAG over the brand knowledge
base (Cortex Search MARKETING_KB) for voice + deck structure; (b) a Cortex
Analyst semantic view (MARKETING_INSIGHTS) over the PRODUCTS, AUDIENCE_SEGMENTS
and PAST_CAMPAIGNS tables, so it grounds real product facts and prior-campaign
channel mixes/results instead of inventing numbers; and (c) two custom deck
tools (generate_html_deck, build_deck) so it produces an actual deck, not just
an outline. Note: Cortex Analyst runs on Snowflake-managed models and needs
cross-region inference; under strict EU-only residency it may be unavailable,
in which case drop the Analyst tool and keep the RAG knowledge base.

**How to use this notebook:** run the cells top to bottom, then follow the Snowsight steps in the final cell to build the agent (and, optionally, the app). Objects are created in the database set in the CONFIG cell (`ANWB_AI_HACKATHON` by default) regardless of the database you pick when importing.

## 1. Setup

This one cell provisions everything the use case needs: it creates the database, warehouse and schema, loads the synthetic sample data, and builds the Cortex Search service (Snowflake's managed hybrid vector + keyword search). It's idempotent - safe to re-run. Run it and wait for the 'Setup complete' message.

In [ ]:
# --- Setup: provisions everything this use case needs (idempotent) ---
from snowflake.snowpark.context import get_active_session
session = get_active_session()

PROVISION = [
 "-- ============================================================================\n-- Challenge 2 - Marketing Agent\n-- ----------------------------------------------------------------------------\n-- WHAT YOU'RE BUILDING: a multi-step marketing agent that turns a product +\n-- audience segment into a structured campaign PLAN and a presentation DECK.\n--\n-- SUCCESS CRITERIA (ANWB): a structured marketing plan; a presentation output;\n-- a demonstrable agent workflow; visible multi-step reasoning and execution.\n--\n-- HOW TO RUN - two ways, same result:\n--   SQL      : paste this file into a Snowsight worksheet and click Run All\n--              (or run: snow sql -c <connection> -f marketing_agent.sql).\n--   Notebook : import marketing_agent.ipynb into Snowsight and Run All.\n-- After Run All, follow the SNOWSIGHT STEPS at the bottom to build the agent\n-- (and, optionally, the app).\n--\n-- Self-contained and idempotent; independent of Challenge 1. The deck defaults\n-- to HTML (no extra packages); a real .pptx is an optional stretch that needs\n-- python-pptx (accept Anaconda terms in Snowsight > Admin > Billing & Terms).\n-- Web Search for live market context is a built-in Cortex Agent tool (not a\n-- custom integration): an admin enables it once for the account\n-- (ALTER ACCOUNT SET ENABLE_CORTEX_WEBSEARCH = true, or AI & ML > Agents >\n-- Settings > Web search), then you add the Web Search tool to the agent.\n--\n-- The agent gets three kinds of grounding: (a) RAG over the brand knowledge\n-- base (Cortex Search MARKETING_KB) for voice + deck structure; (b) a Cortex\n-- Analyst semantic view (MARKETING_INSIGHTS) over the PRODUCTS, AUDIENCE_SEGMENTS\n-- and PAST_CAMPAIGNS tables, so it grounds real product facts and prior-campaign\n-- channel mixes/results instead of inventing numbers; and (c) two custom deck\n-- tools (generate_html_deck, build_deck) so it produces an actual deck, not just\n-- an outline. Note: Cortex Analyst runs on Snowflake-managed models and needs\n-- cross-region inference; under strict EU-only residency it may be unavailable,\n-- in which case drop the Analyst tool and keep the RAG knowledge base.\n-- ============================================================================\n\n-- ============================================================================\n-- CONFIG  --  the only knobs. Edit here to change model, warehouse, or DB.\n-- ============================================================================\nSET db    = 'ANWB_AI_HACKATHON'",
 "-- your DB. For a private copy on a shared\n                                     -- account, set e.g. 'ANWB_AI_HACKATHON_MSA'.\nSET wh    = 'ANWB_AI_HACKATHON_WH'",
 "-- warehouse (created if missing)\nSET model = 'mistral-large2'",
 "-- EU-native (Frankfurt). Alt: 'llama3.3-70b'.\n                                     -- US-only models (Claude/GPT/llama4-maverick)\n                                     -- are NOT reachable under EU-only cross-region.\n\n-- ----------------------------------------------------------------------------\n-- Base (idempotent): database, warehouse, schema\n-- ----------------------------------------------------------------------------\nCREATE DATABASE IF NOT EXISTS IDENTIFIER($db)\n    COMMENT = 'ANWB AI hackathon - Marketing Agent'",
 "CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($wh)\n    WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE\n    INITIALLY_SUSPENDED = TRUE COMMENT = 'Compute for the ANWB AI hackathon'",
 "USE DATABASE IDENTIFIER($db)",
 "CREATE SCHEMA IF NOT EXISTS MARKETING COMMENT = 'Challenge 2 - Marketing Agent'",
 "USE SCHEMA MARKETING",
 "USE WAREHOUSE IDENTIFIER($wh)",
 "-- ----------------------------------------------------------------------------\n-- PRODUCTS — ANWB product/service catalog the agent can build campaigns for\n-- ----------------------------------------------------------------------------\nCREATE OR REPLACE TABLE PRODUCTS (\n    product_id     VARCHAR(10) PRIMARY KEY,\n    name           VARCHAR(100),\n    category       VARCHAR(50),   -- membership | insurance | travel | roadside | shop | energy\n    price_from_eur NUMBER(8,2),\n    price_model    VARCHAR(30),   -- per year | per month | one-off | per trip\n    target_hint    VARCHAR(120),\n    description    VARCHAR(400)\n)",
 "INSERT INTO PRODUCTS VALUES\n('P01','ANWB Lidmaatschap Basis','membership',0.00,'per year','value-seeking members','Entry membership with discounts, the ANWB magazine, and access to member services.'),\n('P02','ANWB Lidmaatschap Plus','membership',33.75,'per year','active families and drivers','Adds Wegenwacht roadside assistance in the Netherlands plus extra member benefits.'),\n('P03','ANWB Wegenwacht Europa Service','roadside',66.00,'per year','frequent European road-trippers','Roadside assistance and recovery across Europe, including towing and replacement transport.'),\n('P04','ANWB Doorlopende Reisverzekering','insurance',5.50,'per month','people who travel more than once a year','Annual travel insurance covering luggage, medical costs, and cancellations for all trips.'),\n('P05','ANWB Kortlopende Reisverzekering','insurance',3.00,'per trip','occasional holidaymakers','Single-trip travel insurance for a specific holiday.'),\n('P06','ANWB Autoverzekering','insurance',9.00,'per month','car owners','Car insurance with member discount and optional roadside cover.'),\n('P07','ANWB Kampeerverzekering','insurance',4.00,'per trip','campers and caravanners','Insurance for tents, caravans, and camping equipment on trips.'),\n('P08','ANWB Camping App & Gids','travel',0.00,'one-off','campers planning European trips','Digital and print camping guide with 9000+ inspected European campsites.'),\n('P09','ANWB Energie','energy',0.00,'per month','cost-conscious households','Green energy for the home with member benefits and transparent tariffs.'),\n('P10','ANWB Rijopleiding','travel',49.00,'one-off','learner drivers 16-24','Driving-lesson packages and theory training via ANWB partners.'),\n('P11','ANWB Webshop Kampeerartikelen','shop',12.50,'one-off','campers and outdoor fans','Camping and travel gear: tents, cool boxes, navigation, safety kits.'),\n('P12','ANWB Fietsverzekering','insurance',3.50,'per month','cyclists and e-bike owners','Bicycle and e-bike insurance against theft and damage, with member discount.')",
 "-- ----------------------------------------------------------------------------\n-- AUDIENCE_SEGMENTS — reusable target personas\n-- ----------------------------------------------------------------------------\nCREATE OR REPLACE TABLE AUDIENCE_SEGMENTS (\n    segment_id    VARCHAR(10) PRIMARY KEY,\n    name          VARCHAR(80),\n    age_range     VARCHAR(20),\n    description   VARCHAR(300),\n    top_channels  VARCHAR(120),\n    key_motivator VARCHAR(160)\n)",
 "INSERT INTO AUDIENCE_SEGMENTS VALUES\n('S01','Young families','30-45','Parents with school-age children planning affordable, safe family holidays by car.','Facebook, Instagram, email, YouTube','Safety, convenience, value for the whole family.'),\n('S02','Active seniors','60-75','Retired members with time and budget for longer European road trips and camping.','Email, print magazine, Facebook','Peace of mind, reliability, being looked after abroad.'),\n('S03','Young adventurers','18-29','Students and young professionals doing budget road trips, festivals, and city breaks.','Instagram, TikTok, YouTube','Freedom, spontaneity, low cost, shareable experiences.'),\n('S04','EV early adopters','35-55','Tech-forward members driving electric vehicles who worry about charging on trips.','Email, YouTube, LinkedIn, Instagram','Range confidence, sustainability, smart planning.'),\n('S05','Caravan enthusiasts','45-65','Experienced caravanners who take multiple camping trips across Europe each year.','Email, print magazine, Facebook groups','Expertise, community, protecting their investment.'),\n('S06','Urban cyclists','25-45','City dwellers relying on bikes and e-bikes for daily transport.','Instagram, email, local out-of-home','Theft protection, convenience, sustainability.')",
 "-- ----------------------------------------------------------------------------\n-- PAST_CAMPAIGNS — prior campaigns for context / few-shot inspiration\n-- ----------------------------------------------------------------------------\nCREATE OR REPLACE TABLE PAST_CAMPAIGNS (\n    campaign_id   VARCHAR(10) PRIMARY KEY,\n    name          VARCHAR(120),\n    product_id    VARCHAR(10),\n    segment_id    VARCHAR(10),\n    year          INT,\n    channel_mix   VARCHAR(120),\n    tagline       VARCHAR(160),\n    result_note   VARCHAR(200)\n)",
 "INSERT INTO PAST_CAMPAIGNS VALUES\n('CM01','Zorgeloos op reis','P04','S01',2024,'Email + Meta + YouTube pre-roll','Zorgeloos op reis, waar je ook heen gaat','+18% policy sign-ups vs. prior year; strong email CTR.'),\n('CM02','Nooit meer stil langs de weg','P03','S02',2024,'Print magazine + email + Facebook','Onderweg pech? Wij staan voor je klaar.','High renewal among seniors; low CAC on email.'),\n('CM03','Laad op, rij door','P03','S04',2025,'YouTube + Instagram + email','Laad op, rij door — heel Europa binnen bereik','Best-performing EV creative; strong under-45 reach.'),\n('CM04','Zomer op de camping','P08','S05',2025,'Email + Facebook groups + print','Jouw perfecte plek staat in de gids','Drove app downloads and guide sales in Q2.'),\n('CM05','Veilig op de fiets','P12','S06',2025,'Instagram + out-of-home + email','Jouw fiets verzekerd, jouw vrijheid gedekt','Grew bike-insurance base among urban cyclists.')",
 "-- ----------------------------------------------------------------------------\n-- KB_DOCUMENTS — brand guidelines, tone of voice, and market context (RAG)\n-- ----------------------------------------------------------------------------\nCREATE OR REPLACE TABLE KB_DOCUMENTS (\n    doc_id     VARCHAR(20) PRIMARY KEY,\n    title      VARCHAR(160),\n    category   VARCHAR(60),\n    source     VARCHAR(60),\n    content    VARCHAR(16000)\n)",
 "INSERT INTO KB_DOCUMENTS (doc_id, title, category, source, content) VALUES\n('MB01','ANWB brand positioning','brand','Brand Guidelines',\n'ANWB is the Royal Dutch Touring Club: a member organisation that has helped people travel safely and enjoyably for over 140 years. The brand promise is \"onderweg thuis\" — feeling looked after wherever you are. Core brand values: reliable, helpful, expert, approachable, and rooted in the Netherlands. ANWB is not a pushy commercial brand; it is a trusted companion. Campaigns should reinforce trust, safety, and genuine care for members rather than hard-selling. The iconic ANWB yellow signals recognition and reassurance on every road.'),\n('MB02','Tone of voice','brand','Brand Guidelines',\n'ANWB speaks in a warm, clear, and helpful voice — like a knowledgeable friend, never corporate or salesy. Guidelines: use plain Dutch (or plain English for international), address the member directly with \"je\", keep sentences short, and lead with the benefit to the member. Be optimistic and reassuring, especially around travel worries. Avoid jargon, fear-mongering, and superlatives that sound like advertising. Humour is welcome when light and human. Always be inclusive and accessible. Example do: \"Onderweg pech? Wij staan voor je klaar.\" Example don''t: \"Profiteer NU van de beste deal ooit!\"'),\n('MB03','Visual identity basics','brand','Brand Guidelines',\n'The primary colour is ANWB yellow (a warm golden yellow) paired with dark blue and plenty of white space. Photography is real and human: members, families, and landscapes, natural light, no overly staged stock imagery. Use the ANWB logo with clear space around it and never recolour it. Typography is clean and legible. Presentations and campaign material should feel open, friendly, and uncluttered, with one clear message per slide or asset.'),\n('MB04','Campaign planning framework','strategy','Marketing Playbook',\n'Every ANWB campaign plan should cover: (1) Objective — what business result (awareness, sign-ups, renewals, app downloads); (2) Target audience — a specific segment with its motivator; (3) Key message — one benefit-led idea in the brand voice; (4) Channel mix — chosen for where the audience is; (5) Content and assets — hero message, supporting proof points, and a call to action; (6) Timeline and phasing; (7) KPIs and how success is measured. Keep the strategy focused: one primary audience and one core message per campaign outperform scattergun approaches.'),\n('MB05','Channel guide','strategy','Marketing Playbook',\n'Channel strengths at ANWB: Email — highest ROI, best for existing members, renewals, and personalised offers. Facebook/Meta — broad reach for families and seniors, good for awareness and retargeting. Instagram — younger members, visual travel inspiration, Stories and Reels. TikTok/YouTube — reach and storytelling for under-30s and EV/tech audiences. Print magazine (the ANWB ledenmagazine) — trusted, high-engagement with seniors and caravanners. Out-of-home — local, tactical awareness (e.g. urban cyclists). Match the channel mix to the segment''s top channels rather than defaulting to everything.'),\n('MB06','Market context: Dutch travel and mobility 2026','market','Market Research',\n'Trends shaping ANWB campaigns in 2026: camping and \"staycation-plus\" trips remain popular as households stay cost-conscious; electric-vehicle adoption keeps rising, and range anxiety on holidays is a top concern; sustainability influences choices, especially among under-45s; members increasingly research and book on mobile. Competitive context: online insurers and price-comparison sites pressure margins, so ANWB leans on trust, service, and roadside expertise as differentiators rather than price. Safety and \"peace of mind\" messaging consistently resonates across segments.'),\n('MB07','Presentation structure for a campaign deck','strategy','Marketing Playbook',\n'A strong campaign presentation follows this slide order: (1) Title — campaign name and one-line vision; (2) The opportunity — the audience insight and business context; (3) Target audience — the segment and its motivator; (4) Strategy — the core idea and why it works; (5) Key messaging — the tagline and supporting messages; (6) Channel plan — the mix and role of each channel; (7) Timeline — phases and key moments; (8) KPIs — how success is measured; (9) Call to action — the ask. Keep one clear idea per slide, use the brand voice, and end with a confident, member-centric close.')",
 "-- ----------------------------------------------------------------------------\n-- Data loaded checkpoint\n-- ----------------------------------------------------------------------------\nSELECT 'Marketing data loaded: '\n    || (SELECT COUNT(*) FROM PRODUCTS) || ' products, '\n    || (SELECT COUNT(*) FROM AUDIENCE_SEGMENTS) || ' segments, '\n    || (SELECT COUNT(*) FROM PAST_CAMPAIGNS) || ' past campaigns, '\n    || (SELECT COUNT(*) FROM KB_DOCUMENTS) || ' KB docs.' AS status",
 "-- ============================================================================\n-- SEARCH  --  hybrid Cortex Search over the brand/marketing knowledge base.\n-- ============================================================================\nEXECUTE IMMEDIATE $$\nBEGIN\n  EXECUTE IMMEDIATE 'CREATE OR REPLACE CORTEX SEARCH SERVICE MARKETING_KB '\n    || 'ON content ATTRIBUTES title, category, source '\n    || 'WAREHOUSE = ' || $wh || ' TARGET_LAG = ''1 hour'' '\n    || 'AS (SELECT doc_id, title, category, source, content FROM KB_DOCUMENTS)';\n  RETURN 'MARKETING_KB search service created';\nEND;\n$$",
 "-- ============================================================================\n-- ANALYST  --  a Cortex Analyst semantic view over the structured tables. This\n-- is what lets the agent ground real numbers: product prices, segment channels,\n-- and (crucially) prior-campaign channel mixes + measured results, instead of\n-- inventing them. Add it to the agent as a Cortex Analyst tool (see SNOWSIGHT\n-- STEPS). Cortex Analyst needs cross-region inference; under strict EU-only\n-- residency it may be unavailable - if so, skip this tool and keep MARKETING_KB.\n-- ============================================================================\nCREATE OR REPLACE SEMANTIC VIEW MARKETING_INSIGHTS\n  TABLES (\n    products  AS PRODUCTS          PRIMARY KEY (product_id)  COMMENT='ANWB products and services',\n    segments  AS AUDIENCE_SEGMENTS PRIMARY KEY (segment_id)  COMMENT='Reusable target audience personas',\n    campaigns AS PAST_CAMPAIGNS    PRIMARY KEY (campaign_id)  COMMENT='Prior ANWB campaigns with channel mix and measured results'\n  )\n  RELATIONSHIPS (\n    campaign_to_product AS campaigns (product_id) REFERENCES products (product_id),\n    campaign_to_segment AS campaigns (segment_id) REFERENCES segments (segment_id)\n  )\n  FACTS (\n    products.price_from AS price_from_eur\n  )\n  DIMENSIONS (\n    products.product_name  AS name           WITH SYNONYMS=('product')            COMMENT='Product name',\n    products.category      AS category        COMMENT='Product category: membership, insurance, travel, roadside, shop, energy',\n    products.price_model   AS price_model     COMMENT='per year, per month, per trip, or one-off',\n    products.target_hint   AS target_hint     COMMENT='Who the product is aimed at',\n    segments.segment_name  AS name            WITH SYNONYMS=('audience','persona') COMMENT='Audience segment name',\n    segments.age_range     AS age_range       COMMENT='Age range of the segment',\n    segments.top_channels  AS top_channels    COMMENT='Marketing channels that work best for this segment',\n    segments.key_motivator AS key_motivator   COMMENT='What primarily motivates this segment',\n    campaigns.campaign_name AS name           COMMENT='Past campaign name',\n    campaigns.campaign_year AS year           COMMENT='Year the campaign ran',\n    campaigns.channel_mix   AS channel_mix    COMMENT='Channels used in the past campaign',\n    campaigns.tagline       AS tagline        COMMENT='Past campaign tagline',\n    campaigns.result_note   AS result_note    COMMENT='Measured outcome of the past campaign'\n  )\n  METRICS (\n    campaigns.campaign_count    AS COUNT(campaigns.campaign_id)   COMMENT='Number of past campaigns',\n    products.avg_price_from_eur AS AVG(products.price_from_eur)   COMMENT='Average starting price in EUR',\n    products.product_count      AS COUNT(products.product_id)     COMMENT='Number of products'\n  )\n  COMMENT='ANWB marketing model: products, audience segments, and past-campaign benchmarks (channel mix + results) for grounding campaign plans'",
 "-- Proof it is queryable (Cortex Analyst uses the same model behind the scenes):\n-- past-campaign channel mixes + results by audience segment.\nSELECT * FROM SEMANTIC_VIEW(\n  MARKETING_INSIGHTS\n  DIMENSIONS segments.segment_name, campaigns.campaign_name, campaigns.channel_mix, campaigns.result_note\n)"
]

for _i, _s in enumerate(PROVISION):
    try:
        session.sql(_s).collect()
    except Exception as _e:
        print(f'[{_i}] {type(_e).__name__}: {_e}')
MODEL = session.sql('SELECT $model').collect()[0][0]
print('Setup complete. Model =', MODEL, '| statements =', len(PROVISION))

## 2. The use case
Run each cell top to bottom; each shows its result.

**Step 1 - Pick the inputs: one product + one audience segment, with plain SQL**

over the PRODUCTS and AUDIENCE_SEGMENTS tables. This is what the agent plans for.

In [ ]:
q = r'''
-- BUILD  --  the multi-step Marketing Agent in SQL. Run these blocks top to
-- bottom. Scenario: a campaign for one product + audience segment. New to
-- Snowflake? Each step notes the Cortex feature it uses and why.
-- ============================================================================

-- Step 1 - Pick the inputs: one product + one audience segment, with plain SQL
-- over the PRODUCTS and AUDIENCE_SEGMENTS tables. This is what the agent plans for.
SELECT p.name AS product, p.description, s.name AS segment, s.key_motivator
FROM PRODUCTS p, AUDIENCE_SEGMENTS s
WHERE p.product_id = 'P01' AND s.segment_id = 'S01'
'''
df = session.sql(q).to_pandas()
df

**Step 2 - Build the campaign plan. RAG again: SEARCH_PREVIEW pulls brand-voice +**

deck-structure docs from the MARKETING_KB Cortex Search service, and AI_COMPLETE
(an LLM in SQL) returns a structured JSON plan grounded in ANWB's guidelines.

In [ ]:
q = r'''
-- Step 2 - Build the campaign plan. RAG again: SEARCH_PREVIEW pulls brand-voice +
-- deck-structure docs from the MARKETING_KB Cortex Search service, and AI_COMPLETE
-- (an LLM in SQL) returns a structured JSON plan grounded in ANWB's guidelines.
SELECT AI_COMPLETE($model,
    'Je bent de Marketing Agent van de ANWB. Maak een campagneplan als JSON met velden: '
 || 'campaign_name, big_idea, audience, key_messages (array), channels (array), kpis (array), '
 || 'slides (array van {title, bullets[]}). Gebruik de merkstem en de deckstructuur uit de kennisbank. '
 || 'Geen uitleg, alleen JSON, geen markdown-fences. '
 || 'Product: ' || (SELECT name||' - '||description FROM PRODUCTS WHERE product_id='P01')
 || '  Doelgroep: ' || (SELECT name||' ('||key_motivator||')' FROM AUDIENCE_SEGMENTS WHERE segment_id='S01')
 || '  Kennisbank (merkstem + deckstructuur): ' || (
        SELECT SUBSTR(TO_JSON(PARSE_JSON(
            SNOWFLAKE.CORTEX.SEARCH_PREVIEW('MARKETING_KB',
              '{"query":"merkstem tone of voice campagne deck structuur presentatie","columns":["title","content"],"limit":4}')
        ):results), 1, 3500)
    )
) AS campaign_plan_json
'''
df = session.sql(q).to_pandas()
df

**Step 3 - Generate the deck. AI_COMPLETE turns the plan into a self-contained HTML**

slide deck (works everywhere, no extra packages). A real .pptx is the optional stretch below.

In [ ]:
q = r'''
-- Step 3 - Generate the deck. AI_COMPLETE turns the plan into a self-contained HTML
-- slide deck (works everywhere, no extra packages). A real .pptx is the optional stretch below.
SELECT AI_COMPLETE($model,
    'Maak een nette, op zichzelf staande HTML-presentatie voor een ANWB-campagne: een <section> per slide, '
 || 'inline CSS, ANWB-geel #FFCC00 als accent. Volg de deckstructuur (titel, opportunity, doelgroep, strategie, '
 || 'messaging, kanalen, timeline, KPIs, call-to-action). Alleen HTML, geen uitleg, geen markdown-fences. '
 || 'Product: ' || (SELECT name||' - '||description FROM PRODUCTS WHERE product_id='P01')
 || '  Doelgroep: ' || (SELECT name||' ('||key_motivator||')' FROM AUDIENCE_SEGMENTS WHERE segment_id='S01')
) AS campaign_deck_html
'''
df = session.sql(q).to_pandas()
df

**DECK TOOLS  --  wrap deck generation as callable procedures so the AGENT can**

produce the actual deliverable (not just a text outline). Add both to the
agent as custom tools (see SNOWSIGHT STEPS):
* generate_html_deck(plan)  -> branded HTML deck. Zero extra packages, works
everywhere (the EU-safe default).
* build_deck(slides, name)  -> real .pptx (optional stretch; needs Anaconda).

In [ ]:
q = r'''
-- ============================================================================
-- DECK TOOLS  --  wrap deck generation as callable procedures so the AGENT can
-- produce the actual deliverable (not just a text outline). Add both to the
-- agent as custom tools (see SNOWSIGHT STEPS):
--   * generate_html_deck(plan)  -> branded HTML deck. Zero extra packages, works
--     everywhere (the EU-safe default).
--   * build_deck(slides, name)  -> real .pptx (optional stretch; needs Anaconda).
-- ============================================================================
CREATE OR REPLACE PROCEDURE generate_html_deck(plan VARIANT)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import html
def run(session, plan):
    if not isinstance(plan, dict):
        plan = {}
    slides = plan.get("slides", []) or []
    title = plan.get("campaign_name") or plan.get("tagline") or "ANWB Campaign"
    css = ("body{font-family:sans-serif;margin:0;background:#f7f7f7}"
           ".slide{border-top:8px solid #FFCC00;margin:16px;padding:24px;border-radius:12px;"
           "background:#fff;box-shadow:0 1px 4px rgba(0,0,0,.08)}"
           "h1{color:#0a2a66}h2{color:#0a2a66;margin-top:0}li{margin:4px 0}")
    parts = ["<style>" + css + "</style>", "<h1>" + html.escape(str(title)) + "</h1>"]
    for s in slides:
        if not isinstance(s, dict):
            continue
        st = html.escape(str(s.get("title", "")))
        lis = "".join("<li>" + html.escape(str(b)) + "</li>" for b in (s.get("bullets", []) or []))
        parts.append("<div class=\"slide\"><h2>" + st + "</h2><ul>" + lis + "</ul></div>")
    return "".join(parts)
$$
'''
df = session.sql(q).to_pandas()
df

**OPTIONAL STRETCH  --  real .pptx via python-pptx.**

Needs Anaconda enabled (Snowsight > Admin > Billing & Terms > accept Anaconda).
If it isn't, this step is skipped and you keep the HTML deck above.

In [ ]:
q = r'''
-- ============================================================================
-- OPTIONAL STRETCH  --  real .pptx via python-pptx.
-- Needs Anaconda enabled (Snowsight > Admin > Billing & Terms > accept Anaconda).
-- If it isn't, this step is skipped and you keep the HTML deck above.
-- ============================================================================
CREATE STAGE IF NOT EXISTS DECKS
'''
df = session.sql(q).to_pandas()
df

In [ ]:
q = "EXECUTE IMMEDIATE $$\nBEGIN\n  CREATE OR REPLACE PROCEDURE build_deck(slides VARIANT, filename STRING)\n  RETURNS STRING\n  LANGUAGE PYTHON\n  RUNTIME_VERSION = '3.11'\n  PACKAGES = ('snowflake-snowpark-python', 'python-pptx')\n  HANDLER = 'run'\n  AS\n  '\nfrom pptx import Presentation\nimport io\ndef run(session, slides, filename):\n    prs = Presentation()\n    for s in slides:\n        slide = prs.slides.add_slide(prs.slide_layouts[1])\n        slide.shapes.title.text = s.get(''title'', '''')\n        tf = slide.placeholders[1].text_frame\n        for i, b in enumerate(s.get(''bullets'', [])):\n            (tf.paragraphs[0] if i == 0 else tf.add_paragraph()).text = b\n    buf = io.BytesIO(); prs.save(buf); buf.seek(0)\n    session.file.put_stream(buf, f''@DECKS/{filename}'', auto_compress=False, overwrite=True)\n    return f''@DECKS/{filename}''\n  ';\n  RETURN 'build_deck procedure created (.pptx available)';\nEXCEPTION WHEN OTHER THEN\n  RETURN 'build_deck skipped - enable Anaconda (python-pptx) for .pptx; the HTML deck already works';\nEND;\n$$"
df = session.sql(q).to_pandas()
df

**Done - a quick readiness summary (database, model, search service, deck options).**

In [ ]:
q = r'''
-- Done - a quick readiness summary (database, model, search service, deck options).
SELECT 'Marketing Agent ready in ' || $db || '.MARKETING  |  model=' || $model
    || '  |  search=MARKETING_KB  |  analyst=MARKETING_INSIGHTS'
    || '  |  deck tools: generate_html_deck (default) + build_deck (.pptx stretch)'
    AS status
'''
df = session.sql(q).to_pandas()
df

## 3. Do this in Snowsight (agent, app, observability)

```
SNOWSIGHT STEPS  --  the parts you do in the Snowsight UI (clicks, not SQL).
Do these after Run All above succeeds.
STEP A - Build the Marketing Agent
  1. Left nav > AI & ML > Agents > "+ Agent" (Create agent).
  2. Schema ANWB_AI_HACKATHON.MARKETING; name it MARKETING_AGENT; Create.
  3. Open the agent > Tools and add:
       a. Cortex Search  ->  ANWB_AI_HACKATHON.MARKETING.MARKETING_KB
            (brand voice + deck structure; the RAG index)
       b. Cortex Analyst ->  ANWB_AI_HACKATHON.MARKETING.MARKETING_INSIGHTS
            (real product facts + past-campaign channel mixes/results, so the
             agent grounds numbers instead of inventing them). Skip this one
             only if Cortex Analyst is unavailable under EU-only residency.
       c. Custom tool    ->  generate_html_deck  (procedure; input: plan VARIANT)
            so the agent returns an actual branded HTML deck.
       d. Custom tool    ->  build_deck  (procedure; inputs: slides VARIANT,
            filename STRING) for a real .pptx  (optional; needs Anaconda).
       e. (Optional) Web Search for live market context - needs
            ENABLE_CORTEX_WEBSEARCH enabled by an admin (see header note).
  4. Model = mistral-large2. Instructions (paste this):
       "You are the ANWB Marketing Agent. Given a product + audience, produce
        (1) a structured campaign plan and (2) a presentation outline.
        GROUNDING & CITATIONS: use the MARKETING_INSIGHTS Analyst tool for real
        product facts and prior-campaign channel mixes/results, and the
        MARKETING_KB search tool for brand voice + deck structure; cite the
        source of each fact (e.g. 'past campaign CM02' or the KB doc title).
        ILLUSTRATIVE FIGURES: any number you did NOT get from the Analyst tool
        or the knowledge base - budget splits, ROI/CPA, audience sizes, KPI
        targets - must be labelled '(illustrative)'; never present an invented
        number as verified. DECK: call generate_html_deck (or build_deck for
        .pptx) to produce the actual deck. Keep ANWB voice: warm, clear,
        plain Dutch 'je', benefit-first, reassuring - never hard-sell."
  5. Sample questions (add under the agent's sample/onboarding questions):
       - "Maak een campagne voor ANWB Wegenwacht Europa voor actieve senioren."
       - "Bouw een najaarscampagne voor Wegenwacht gericht op jonge gezinnen."
       - "Welke kanalenmix werkte in eerdere campagnes voor jonge gezinnen?"
       - "Campagne voor de ANWB Doorlopende Reisverzekering voor 25-35-jarigen."
       - "Promoot laadpas/EV-laden bij leden die net een elektrische auto kochten."
  6. Save, open the chat, and ask one of the sample questions.
     Success = a structured plan + a generated deck, with real facts cited and
     any assumed numbers marked (illustrative), showing visible multi-step
     tool use (Analyst + Search + deck tool).

STEP A-alt (optional) - create the agent with SQL instead of clicking.
Agent creation is GA; this makes the enriched agent reproducible/testable. The
manual UI build above stays the primary path. Uncomment and run:
  /*
  CREATE OR REPLACE AGENT ANWB_AI_HACKATHON.MARKETING.MARKETING_AGENT
    WITH PROFILE='{"display_name":"ANWB Marketing Agent"}'
    COMMENT='ANWB campaign-plan + deck agent (RAG + Analyst + deck tools)'
    FROM SPECIFICATION $spec$
    {
      "models": {"orchestration": "mistral-large2"},
      "instructions": {"orchestration": "<paste the STEP A step-4 instructions>"},
      "tools": [
        {"tool_spec": {"type": "cortex_search",             "name": "Marketing_KB"}},
        {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "Marketing_Insights"}},
        {"tool_spec": {"type": "generic", "name": "generate_html_deck",
           "description": "Render a branded HTML deck from a plan JSON",
           "input_schema": {"type":"object","properties":{"plan":{"type":"object"}},"required":["plan"]}}}
      ],
      "tool_resources": {
        "Marketing_KB":       {"search_service": "ANWB_AI_HACKATHON.MARKETING.MARKETING_KB", "max_results": 4},
        "Marketing_Insights": {"semantic_view": "ANWB_AI_HACKATHON.MARKETING.MARKETING_INSIGHTS",
                               "execution_environment": {"type": "warehouse", "warehouse": "ANWB_AI_HACKATHON_WH"}},
        "generate_html_deck": {"identifier": "ANWB_AI_HACKATHON.MARKETING.GENERATE_HTML_DECK", "type": "procedure",
                               "execution_environment": {"type": "warehouse", "warehouse": "ANWB_AI_HACKATHON_WH"}}
      }
    }
    $spec$;
  */

STEP B (optional) - Ship the app
  Projects > Streamlit > "+ Streamlit App"; warehouse ANWB_AI_HACKATHON_WH, database
  ANWB_AI_HACKATHON, schema MARKETING. Paste app.py from this folder and Run.

STEP C (optional) - Real .pptx deck
  Enable Anaconda (Admin > Billing & Terms), then
    CALL MARKETING.build_deck(<slides JSON>, 'deck.pptx');
  and download it from the DECKS stage (Data > Databases > ... > DECKS).

STEP D (optional) - Inspect what the agent did (AI Observability)
  AI & ML > Agents > MARKETING_AGENT > Monitoring: traces, tool calls, latency, tokens.
```